In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print("PyTorch version:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
stock = "ABUK"

stock_path = Path(f"../../data/egx/{stock}.csv")

stock_df = pd.read_csv(stock_path)

print("Stock:", stock)
print("Shape:", stock_df.shape)

display(stock_df.head())

In [ ]:
stock_df["date"] = pd.to_datetime(stock_df["date"])

stock_df = (
    stock_df
    .sort_values("date")
    .reset_index(drop=True)
)

stock_df["return"] = stock_df["close"].pct_change()

stock_df = (
    stock_df
    .dropna(subset=["return"])
    .reset_index(drop=True)
)

In [ ]:
for lag in range(1, 6):
    stock_df[f"return_lag_{lag}"] = (
        stock_df["return"].shift(lag)
    )

stock_df = (
    stock_df
    .dropna()
    .reset_index(drop=True)
)

features = [
    "return_lag_1",
    "return_lag_2",
    "return_lag_3",
    "return_lag_4",
    "return_lag_5"
]

X = stock_df[features]
y = stock_df["return"]

In [ ]:
split_index = int(len(X) * 0.70)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

In [ ]:
X_train_seq = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    X_train_scaled.shape[1],
    1
)

X_test_seq = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    X_test_scaled.shape[1],
    1
)

print("Train:", X_train_seq.shape)
print("Test :", X_test_seq.shape)

In [ ]:
class LSTMModel(nn.Module):

    def __init__(self, input_size=1, hidden_size=64):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):

        out, _ = self.lstm(x)

        last_output = out[:, -1, :]

        prediction = self.fc(last_output)

        return prediction

In [ ]:
X_train_tensor = torch.tensor(
    X_train_seq,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_seq,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
).view(-1, 1)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
).view(-1, 1)

print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)

print("X_test :", X_test_tensor.shape)
print("y_test :", y_test_tensor.shape)

In [ ]:
batch_size = 32

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Number of training batches:", len(train_loader))
print("Number of testing batches:", len(test_loader))

In [ ]:
class LSTMModel(nn.Module):

    def __init__(self, input_size=1, hidden_size=64):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        # x shape:
        # (batch, sequence_length, input_size)

        out, _ = self.lstm(x)

        # Take the output from the last timestep
        last_output = out[:, -1, :]

        prediction = self.fc(last_output)

        return prediction

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = LSTMModel(
    input_size=1,
    hidden_size=64
).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)
print("\nDevice:", device)

In [ ]:
num_epochs = 100

train_losses = []
test_losses = []

for epoch in range(num_epochs):

    # =====================
    # Training
    # =====================

    model.train()

    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        predictions = model(X_batch)

        loss = criterion(
            predictions,
            y_batch
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # =====================
    # Testing
    # =====================

    model.eval()

    test_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predictions = model(X_batch)

            loss = criterion(
                predictions,
                y_batch
            )

            test_loss += loss.item()

    test_loss /= len(test_loader)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch [{epoch+1:3d}/{num_epochs}] "
            f"Train Loss: {train_loss:.6f} "
            f"Test Loss: {test_loss:.6f}"
        )

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    train_losses,
    label="Train Loss"
)

plt.plot(
    test_losses,
    label="Test Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("LSTM Training and Test Loss")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
model.eval()

train_predictions = []
test_predictions = []

with torch.no_grad():

    for X_batch, _ in train_loader:

        X_batch = X_batch.to(device)

        predictions = model(X_batch)

        train_predictions.extend(
            predictions.cpu().numpy().flatten()
        )

    for X_batch, _ in test_loader:

        X_batch = X_batch.to(device)

        predictions = model(X_batch)

        test_predictions.extend(
            predictions.cpu().numpy().flatten()
        )

train_predictions = np.array(train_predictions)
test_predictions = np.array(test_predictions)

print("Train predictions:", train_predictions.shape)
print("Test predictions :", test_predictions.shape)

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    y_train.values,
    label="Actual"
)

plt.plot(
    train_predictions,
    label="Predicted"
)

plt.xlabel("Time")
plt.ylabel("Return")
plt.title("LSTM — Train: Actual vs Predicted Returns")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    y_test.values,
    label="Actual"
)

plt.plot(
    test_predictions,
    label="Predicted"
)

plt.xlabel("Time")
plt.ylabel("Return")
plt.title("LSTM — Test: Actual vs Predicted Returns")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
lstm_mse = mean_squared_error(
    y_test,
    test_predictions
)

lstm_mae = mean_absolute_error(
    y_test,
    test_predictions
)

lstm_r2 = r2_score(
    y_test,
    test_predictions
)

directional_accuracy = np.mean(
    np.sign(y_test.values) ==
    np.sign(test_predictions)
)

print("LSTM Results")
print("=" * 30)

print(f"Test MSE: {lstm_mse:.6f}")
print(f"Test MAE: {lstm_mae:.6f}")
print(f"Test R²: {lstm_r2:.6f}")
print(
    f"Directional Accuracy: "
    f"{directional_accuracy:.2%}"
)

In [ ]:
class GRUModel(nn.Module):

    def __init__(self, input_size=1, hidden_size=64):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        # x shape:
        # (batch, sequence_length, input_size)

        out, _ = self.gru(x)

        # Take the output from the last timestep
        last_output = out[:, -1, :]

        prediction = self.fc(last_output)

        return prediction

In [ ]:
gru_model = GRUModel(
    input_size=1,
    hidden_size=64
).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    gru_model.parameters(),
    lr=0.001
)

print(gru_model)

In [ ]:
num_epochs = 100

gru_train_losses = []
gru_test_losses = []

for epoch in range(num_epochs):

    # =====================
    # Training
    # =====================

    gru_model.train()

    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        predictions = gru_model(X_batch)

        loss = criterion(
            predictions,
            y_batch
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # =====================
    # Testing
    # =====================

    gru_model.eval()

    test_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predictions = gru_model(X_batch)

            loss = criterion(
                predictions,
                y_batch
            )

            test_loss += loss.item()

    test_loss /= len(test_loader)

    gru_train_losses.append(train_loss)
    gru_test_losses.append(test_loss)

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch [{epoch+1:3d}/{num_epochs}] "
            f"Train Loss: {train_loss:.6f} "
            f"Test Loss: {test_loss:.6f}"
        )

In [ ]:
gru_model.eval()

gru_train_predictions = []
gru_test_predictions = []

with torch.no_grad():

    for X_batch, _ in train_loader:

        X_batch = X_batch.to(device)

        predictions = gru_model(X_batch)

        gru_train_predictions.extend(
            predictions.cpu().numpy().flatten()
        )

    for X_batch, _ in test_loader:

        X_batch = X_batch.to(device)

        predictions = gru_model(X_batch)

        gru_test_predictions.extend(
            predictions.cpu().numpy().flatten()
        )

gru_train_predictions = np.array(gru_train_predictions)
gru_test_predictions = np.array(gru_test_predictions)

print("Train predictions:", gru_train_predictions.shape)
print("Test predictions :", gru_test_predictions.shape)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    gru_train_losses,
    label="Train Loss"
)

plt.plot(
    gru_test_losses,
    label="Test Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("GRU Training and Test Loss")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    y_train.values,
    label="Actual"
)

plt.plot(
    gru_train_predictions,
    label="Predicted"
)

plt.xlabel("Time")
plt.ylabel("Return")
plt.title("GRU — Train: Actual vs Predicted Returns")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    y_test.values,
    label="Actual"
)

plt.plot(
    gru_test_predictions,
    label="Predicted"
)

plt.xlabel("Time")
plt.ylabel("Return")
plt.title("GRU — Test: Actual vs Predicted Returns")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
gru_mse = mean_squared_error(
    y_test,
    gru_test_predictions
)

gru_mae = mean_absolute_error(
    y_test,
    gru_test_predictions
)

gru_r2 = r2_score(
    y_test,
    gru_test_predictions
)

gru_directional_accuracy = np.mean(
    np.sign(y_test.values) ==
    np.sign(gru_test_predictions)
)

print("GRU Results")
print("=" * 30)

print(f"Test MSE: {gru_mse:.6f}")
print(f"Test MAE: {gru_mae:.6f}")
print(f"Test R²: {gru_r2:.6f}")
print(
    f"Directional Accuracy: "
    f"{gru_directional_accuracy:.2%}"
)

In [ ]:
comparison = pd.DataFrame({
    "Model": ["MLP", "LSTM", "GRU"],
    "Test MSE": [
        0.001049,
        lstm_mse,
        gru_mse
    ],
    "Test MAE": [
        0.021692,
        lstm_mae,
        gru_mae
    ],
    "Test R2": [
        -0.543638,
        lstm_r2,
        gru_r2
    ],
    "Directional Accuracy": [
        0.4988,
        directional_accuracy,
        gru_directional_accuracy
    ]
})

comparison

In [ ]:
comparison.sort_values(
    "Test MSE"
).reset_index(drop=True)